本文档用于协助开发Behavior Prompt在Tbot上，主要是在开发过程中，验证每个模块的功能性，支持开发进行。

例如，加载LerobotDataset的扩展类，检测是否可用于数据加载，以及实现了数据处理管道

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
delta_timestamps = {
    "action": [i / 30 for i in range(10)],
    "observation.images.cam_high": [-0.5, 0.0, 0.5],
    "observation.images.cam_left_wrist": [-0.5, 0.0, 0.5],
    "observation.images.cam_right_wrist": [-0.5, 0.0, 0.5],
}
ds = LeRobotDataset('/vla/workspace/data/robotwin2.0/hanging_mug/aloha-agilex_clean_50',delta_timestamps=delta_timestamps)
ds1 = LeRobotDataset('/vla/workspace/data/robotwin2.0/hanging_mug/aloha-agilex_clean_50')

In [ ]:
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptLeRobotDataset,BehaviorPromptConfig
config = BehaviorPromptConfig()
config.prompt_action_chunk_size = 10
config.max_prompt_chunks = None
config.num_chunks = 10
config.same_episode_policy = "avoid"
config.seed = 0

bp_ds = BehaviorPromptLeRobotDataset.with_default_transforms_v2(ds, ds1, config)
sample = bp_ds[0]
bp_ds

In [ ]:
for i, step in enumerate(bp_ds.transform.transforms):
    print(f"data process step:  [{i}] {step.__class__.__name__}")

In [ ]:
sample['behavior_prompt']['action'].shape

#### 验证 BPObsEncoder 能否编码

In [ ]:
# 验证 BPObsEncoder 能否编码 sample['behavior_prompt']
# 说明：这里用 timm resnet18 + 小维度做快速 smoke；正式模型可改回 token_dim=768/output_dim=2048。
import importlib.util
from pathlib import Path
import sys
import torch
from lerobot.utils.constants import OBS_IMAGES

# 避免 lerobot.policies.__init__ 一次性导入其他策略依赖；这里只验证本文件。
module_path = Path("/vla/workspace/my_tbot/src/lerobot/policies/BP_TBot_v2/bp_transformer_obs_encoder.py")
spec = importlib.util.spec_from_file_location("bp_transformer_obs_encoder_v2_smoke", module_path)
bp_encoder_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = bp_encoder_module
spec.loader.exec_module(bp_encoder_module)
BPObsEncoder = bp_encoder_module.BPObsEncoder
BPTransformerObsEncoder = bp_encoder_module.BPTransformerObsEncoder

bp = sample["behavior_prompt"]
bp_num_chunks = int(bp["mask"].shape[0])
action_chunk_size = int(bp["action"].shape[-2])
state_dim = int(bp["state"].shape[-1])
action_dim = int(bp["action"].shape[-1])
image_keys = [f"{OBS_IMAGES}.image{i}" for i in range(3)]

chunk_encoder = BPTransformerObsEncoder(
    image_keys=image_keys, # 三路图像名称
    vision_model_name="resnet18", # 视觉骨干，原BPP为"vit_base_patch16_clip_224.openai"
    pretrained=False, # 是否加载预训练视觉权重
    token_dim=768, # 每个 image token: 768
    output_dim=2048, # chunk_fusion  将 3 +1 +1，768 压缩为 2048
    state_dim=state_dim,
    action_dim=action_dim,
    action_chunk_size=action_chunk_size,
    image_feature_aggregation="mean",
)
bp_encoder = BPObsEncoder(
    chunk_encoder=chunk_encoder,
    max_num_chunks=bp_num_chunks,
)

bp_encoder.eval()
with torch.no_grad():
    encoded = bp_encoder(bp, return_dict=True)

print("bp_num_chunks:", bp_num_chunks)
print("state:", tuple(bp["state"].shape))
print("action:", tuple(bp["action"].shape))
print("mask:", tuple(bp["mask"].shape))
print("chunk_tokens:", tuple(encoded.chunk_tokens.shape))
print("modality_tokens:", tuple(encoded.modality_tokens.shape))
print("chunk_indices:", encoded.chunk_indices)
print("encoded mask:", encoded.mask)

### 推理与训练


In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate
# 用bp_ds[0],bp_ds[1] 构建batch size =2 的batch
samples = [bp_ds[0],bp_ds[1]]
batch = default_collate(samples)

import torch
def move_to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, dict):
        return {k: move_to_device(v, device) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to_device(v, device) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to_device(v, device) for v in x)
    return x
batch_for_forward = move_to_device(batch, "cuda")

### 加载policy

In [ ]:
# 正式规模加载 tbot_bp policy：BPObsEncoder 会作为 policy.model 的子模块进入同一个 checkpoint。
# 这里不加载 tbot_base/BP checkpoint，只加载可用的 Qwen3-VL 与 Cosmos tokenizer 模块。
import torch
from lerobot.policies.BP_TBot_v2.configuration_bp_tbot import BPTBotV2Config
from lerobot.policies.BP_TBot_v2.modeling_bp_tbot import TBotBPPolicy

cfg = BPTBotV2Config()
cfg.pretrained_path = None
cfg.device = "cuda"
cfg.dtype = "bfloat16"
cfg.qwen3_vl_pretrained_path = "/vla/workspace/models/Qwen3-VL-2B-Instruct"
cfg.cosmos_tokenizer_path_or_name = "/vla/workspace/models/Cosmos-Tokenizer-CI8x8"

# 和当前 bp_ds 构造保持一致：notebook 前面设置了 prompt_action_chunk_size=10、num_chunks=10。
# cfg.bp_num_chunks = int(batch_for_forward["behavior_prompt"]["mask"].shape[1])
# cfg.bp_action_chunk_size = int(batch_for_forward["behavior_prompt"]["action"].shape[-2])
cfg.bp_num_chunks = 10
cfg.bp_action_chunk_size = 10
cfg.chunk_size=10 # 默认为50
cfg.bp_vision_model_name = "vit_base_patch16_clip_224.openai"
cfg.bp_vision_pretrained = False
cfg.bp_token_dim = 768
cfg.bp_image_feature_aggregation = "cls"

# 先关闭 DA3 teacher，避免本轮验证额外加载第三个外部大模型；BP/TBot 主前向仍是正式 hidden size。
cfg.lambda_3d = 0.0

policy = TBotBPPolicy(cfg)
policy.train()
print(policy.name)
print("bp_obs_encoder in checkpoint:", any(k.startswith("model.bp_obs_encoder.") for k in policy.state_dict()))
print("bp_num_chunks:", cfg.bp_num_chunks)
print("bp_action_chunk_size:", cfg.bp_action_chunk_size)
print("bp output dim:", policy.model.bp_obs_encoder.chunk_dim)


### Forward

In [ ]:
# 真实 batch 前向：返回 total loss 和各项 loss 日志。
# 注意：这是正式模型规模，显存压力明显高于前面的 BPObsEncoder 单模块验证。
torch.cuda.empty_cache()
policy.train()
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    loss, loss_dict = policy(batch_for_forward)

print("loss:", float(loss.detach().cpu()))
for key, value in loss_dict.items():
    if key.startswith("loss_action_dim"):
        continue
    print(f"{key}: {value}")

# 反向也走一下，确认 BPObsEncoder 在同一个计算图里。
loss.backward()
print("backward ok")


### 配置policy，从Tbot 或 已有权重中加载模型

In [1]:
# 独立单元：配置 tbot_bp policy，并从 TBot base 或已有 tbot_bp 权重加载。
# 只改下面这些大写常量即可。
from pathlib import Path
import torch

from lerobot.policies.BP_TBot_v2.configuration_bp_tbot import BPTBotV2Config
from lerobot.policies.BP_TBot_v2.modeling_bp_tbot import TBotBPPolicy

LOAD_MODE = "TBOT_BASE"  # 可选: "TBOT_BASE" / "TBOT_BP" / "SCRATCH"
TBOT_BASE_DIR = Path("/vla/workspace/models/tbot_base")
TBOT_BP_DIR = Path("/vla/workspace/models/tbot_bp")
SAVE_DIR = Path("/vla/workspace/models/tbot_bp_test_saved")

QWEN3_VL_DIR = Path("/vla/workspace/models/Qwen3-VL-2B-Instruct")
COSMOS_DIR = Path("/vla/workspace/models/Cosmos-Tokenizer-CI8x8")
DEVICE = "cuda"
DTYPE = "bfloat16"
STRICT_LOAD = False

# 如果这个单元独立运行，不依赖前面 batch，则用正式训练默认规模。
BP_NUM_CHUNKS = 10 # 接受的chunks个数 即K ； 每个样本取 10 个 BP chunk
BP_ACTION_CHUNK_SIZE = 50 # 每个 BP chunk 里包含 BP_ACTION_CHUNK_SIZE- 10 步 action
BP_VISION_MODEL_NAME = "vit_base_patch16_clip_224.openai"
BP_VISION_PRETRAINED = False
BP_TOKEN_DIM = 768
BP_IMAGE_FEATURE_AGGREGATION = "cls"

# 本轮只是配置/存储权重，不额外加载 DA3 teacher；后续训练可在配置文件里打开。
ENABLE_3D_QUERIES = True
LAMBDA_3D = 0.0

cfg = BPTBotV2Config()
cfg.pretrained_path = None
cfg.device = DEVICE
cfg.dtype = DTYPE
cfg.qwen3_vl_pretrained_path = str(QWEN3_VL_DIR)
cfg.cosmos_tokenizer_path_or_name = str(COSMOS_DIR)
cfg.bp_num_chunks = BP_NUM_CHUNKS
cfg.bp_action_chunk_size = BP_ACTION_CHUNK_SIZE
cfg.bp_vision_model_name = BP_VISION_MODEL_NAME
cfg.bp_vision_pretrained = BP_VISION_PRETRAINED
cfg.bp_token_dim = BP_TOKEN_DIM
cfg.bp_image_feature_aggregation = BP_IMAGE_FEATURE_AGGREGATION
cfg.enable_3d_queries = ENABLE_3D_QUERIES
cfg.lambda_3d = LAMBDA_3D

if LOAD_MODE == "TBOT_BASE":
    LOAD_DIR = TBOT_BASE_DIR
    policy_for_save = TBotBPPolicy.from_pretrained(LOAD_DIR, config=cfg, strict=STRICT_LOAD)
elif LOAD_MODE == "TBOT_BP":
    LOAD_DIR = TBOT_BP_DIR
    policy_for_save = TBotBPPolicy.from_pretrained(LOAD_DIR, config=cfg, strict=STRICT_LOAD)
elif LOAD_MODE == "SCRATCH":
    LOAD_DIR = None
    policy_for_save = TBotBPPolicy(cfg)
else:
    raise ValueError(f"Unsupported LOAD_MODE: {LOAD_MODE}")

policy_for_save.eval()
print("LOAD_MODE:", LOAD_MODE)
print("LOAD_DIR:", LOAD_DIR)
print("SAVE_DIR:", SAVE_DIR)
print("policy name:", policy_for_save.name)
print("device:", cfg.device, "dtype:", cfg.dtype)
print("bp_obs_encoder in state_dict:", any(k.startswith("model.bp_obs_encoder.") for k in policy_for_save.state_dict()))
print("num state_dict keys:", len(policy_for_save.state_dict()))

/vla/.conda/miniconda3/envs/mytbot/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading weights from local directory


LOAD_MODE: TBOT_BASE
LOAD_DIR: /vla/workspace/models/tbot_base
SAVE_DIR: /vla/workspace/models/tbot_bp_test_saved
policy name: tbot_bp
device: cuda dtype: bfloat16
bp_obs_encoder in state_dict: True
num state_dict keys: 1963


###  存储模型，包括config.json文件

In [2]:
# 独立单元：保存 policy 权重与 config.json。
# 依赖上一单元得到的 policy_for_save / SAVE_DIR；如果你重启 kernel，请先运行上一单元。
from pathlib import Path
from huggingface_hub.constants import SAFETENSORS_SINGLE_FILE

SAVE_DIR = Path(SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 使用 LeRobot PreTrainedPolicy 的保存路径：会写 config.json + model.safetensors。
policy_for_save._save_pretrained(SAVE_DIR)

saved_files = sorted(p.name for p in SAVE_DIR.iterdir())
print("saved to:", SAVE_DIR)
print("files:", saved_files)
print("has config.json:", (SAVE_DIR / "config.json").exists())
print(f"has {SAFETENSORS_SINGLE_FILE}:", (SAVE_DIR / SAFETENSORS_SINGLE_FILE).exists())

# 简单确认保存出来的权重包含 BPObsEncoder；不加载整模型，只读 safetensors key。
from safetensors.torch import safe_open
weight_path = SAVE_DIR / SAFETENSORS_SINGLE_FILE
with safe_open(weight_path, framework="pt", device="cpu") as f:
    keys = list(f.keys())
    bp_keys = [k for k in keys if k.startswith("model.bp_obs_encoder.")]
print("num saved keys:", len(keys))
print("num bp_obs_encoder keys:", len(bp_keys))
print("first bp key:", bp_keys[0] if bp_keys else None)


saved to: /vla/workspace/models/tbot_bp_test_saved
files: ['config.json', 'model.safetensors']
has config.json: True
has model.safetensors: True
num saved keys: 1458
num bp_obs_encoder keys: 170
first bp key: model.bp_obs_encoder.chunk_encoder.action_proj.0.bias
